# Fundamentals of Machine Learning – Programming Assignment
## Heart Disease Prediction using Logistic Regression and Support Vector Machine (SVM)

**Dataset:** Cleveland Heart Disease Dataset
**Source:** https://archive.ics.uci.edu/dataset/45/heart+disease
**Algorithms:** Logistic Regression | Support Vector Machine (SVM)
**Language:** Python | Environment: Jupyter Notebook

---
## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn: preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Scikit-learn: models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Scikit-learn: evaluation
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)

---
## 2. Load the Dataset

The **Cleveland Heart Disease Dataset** contains 303 patient records with 13 clinical features.
The target variable indicates the presence (1) or absence (0) of heart disease.

**Dataset Link:** https://archive.ics.uci.edu/dataset/45/heart+disease

In [ ]:
# Load dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

column_names = [
    'age',        # Age in years
    'sex',        # 1 = male, 0 = female
    'cp',         # Chest pain type (0-3)
    'trestbps',   # Resting blood pressure (mm Hg)
    'chol',       # cholesterol (mg/dl)
    'fbs',        # Fasting blood sugar > 120 mg/dl (1 = true, 0 = false)
    'restecg',    # Resting ECG results (0-2)
    'thalach',    # Maximum heart rate achieved
    'exang',      # Exercise-induced angina (1 = yes, 0 = no)
    'oldpeak',    # ST depression induced by exercise
    'slope',      # Slope of peak exercise ST segment (0-2)
    'ca',         # Number of major vessels (0-3) colored by fluoroscopy
    'thal',       # Thalassemia type (3 = normal, 6 = fixed defect, 7 = reversible defect)
    'target'      # Diagnosis (0 = no disease, 1-4 = disease present)
]

# Replace '?' with NaN on load
df = pd.read_csv(url, names=column_names, na_values='?')

# Convert target to binary:  1 = disease0 = no disease,
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)

print(f"Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print("=== Dataset Info ===")
df.info()
print("\n=== Statistical Summary ===")
df.describe()

In [ ]:
# Check for missing values
print("=== Missing Values ===")
missing = df.isnull().sum()
print(missing[missing > 0])
print(f"\nTotal missing: {df.isnull().sum().sum()}")

In [ ]:
# Target class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
target_counts = df['target'].value_counts()
axes[0].bar(['No Disease (0)', 'Disease (1)'], target_counts.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Class Distribution', fontsize=14)
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(target_counts.values, labels=['No Disease', 'Disease'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Class Distribution (%)', fontsize=14)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Class distribution:\n{target_counts}")

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 9))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Age distribution by target
plt.figure(figsize=(10, 5))
df[df['target']==0]['age'].hist(bins=20, alpha=0.7, label='No Disease', color='#2ecc71')
df[df['target']==1]['age'].hist(bins=20, alpha=0.7, label='Disease', color='#e74c3c')
plt.xlabel('Age')
plt.ylabel('Count')
plt.title('Age Distribution by Heart Disease Status')
plt.legend()
plt.tight_layout()
plt.savefig('age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Data Preprocessing

In [ ]:
# Step 1: Handle missing values using median imputation
imputer = SimpleImputer(strategy='median')
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

print(f"Missing values after imputation: {df_imputed.isnull().sum().sum()}")

# Step 2: Separate features and target
X = df_imputed.drop('target', axis=1)
y = df_imputed['target']

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape:   {y.shape}")
print(f"Feature names:  {list(X.columns)}")

In [ ]:
# Step 3: Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set:  {X_train.shape[0]} samples")
print(f"Test set:      {X_test.shape[0]} samples")
print(f"Train class distribution: {dict(y_train.value_counts())}")
print(f"Test class distribution:  {dict(y_test.value_counts())}")

In [ ]:
# Step 4: Feature Scaling (StandardScaler - important for SVM and Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # Fit on train, transform train
X_test_scaled  = scaler.transform(X_test)         # Only transform test (no data leakage)

print("✅ Feature scaling complete.")
print(f"Mean of scaled training data (should be ~0): {X_train_scaled.mean():.4f}")
print(f"Data PreprocessingStd  of scaled training data (should be ~1): {X_train_scaled.std():.4f}")